# 01 - Deploy Infrastructure

Deploys `infra/main.bicep` and writes deployment outputs to `env/.env` so the later notebooks can run without manual value copy/paste.

**Estimated time:** 6-10 minutes

---

## What gets deployed

- Azure AI Services account (`kind: AIServices`, `disableLocalAuth: false` — scoped exception, see README)
- Foundry project + `gpt-5.4-mini` `GlobalStandard` deployment
- Azure Container Registry (Basic, admin off; Function App pulls with MI)
- Elastic Premium (EP1) Linux plan
- Custom-container Function App (empty image on first deploy — notebook 02 builds and swaps it)
- App Insights + role assignments (Function MI → ACR AcrPull, Storage Blob Data Owner)

---

## Step 1 - Set deployment variables


In [ ]:
RESOURCE_GROUP = "rg-copilotcli01-foundry"
LOCATION = "eastus2"
DEMO_ID = "copilotcli01"

print(f"Resource group : {RESOURCE_GROUP}")
print(f"Location       : {LOCATION}")
print(f"Demo ID        : {DEMO_ID}")

## Step 2 - Create the resource group

In [ ]:
!az group create \
    --name {RESOURCE_GROUP} \
    --location {LOCATION} \
    --tags SecurityControl=Ignore \
    --output table


## Step 3 - Deploy Bicep template

The Function App is created pointing at an ACR image tag that does not exist yet. It will report a container-start failure until notebook 02 runs `az acr build` and restarts the app. This is expected.

In [ ]:
import json
import re
import subprocess
import sys
import time
from datetime import datetime
from pathlib import Path


def log(msg: str) -> None:
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}", flush=True)


log("Cell start — importing helpers")

demo_root = Path("..").resolve()
if str(demo_root) not in sys.path:
    sys.path.insert(0, str(demo_root))

from app.notebook_helpers import resolve_az_cli

az_cmd = resolve_az_cli()
if not az_cmd:
    raise RuntimeError("Azure CLI not found. Install it or add it to PATH.")
log(f"Resolved az CLI: {az_cmd}")

DEPLOYMENT_NAME = "copilotcli01-main"
POLL_INTERVAL_SECONDS = 5

log("Looking up signed-in user object ID (az ad signed-in-user show)...")
deployer_object_id_result = subprocess.run(
    [az_cmd, "ad", "signed-in-user", "show", "--query", "id", "-o", "tsv"],
    capture_output=True,
    text=True,
)
deployer_object_id = deployer_object_id_result.stdout.strip()
if deployer_object_id:
    log(f"Deployer principal ID for AcrPush RBAC: {deployer_object_id}")
else:
    log("Could not detect signed-in user object ID; skipping deployerPrincipalId parameter.")


def extract_soft_deleted_account_name(text: str) -> str:
    match = re.search(r"/Microsoft\.CognitiveServices/accounts/([^'\s\"]+)", text)
    return match.group(1) if match else ""


def submit_deployment(include_deployer_principal: bool):
    parameters = [
        f"location={LOCATION}",
        f"demoId={DEMO_ID}",
    ]
    if include_deployer_principal and deployer_object_id:
        parameters.append(f"deployerPrincipalId={deployer_object_id}")
    cmd = [
        az_cmd,
        "deployment",
        "group",
        "create",
        "--resource-group",
        RESOURCE_GROUP,
        "--name",
        DEPLOYMENT_NAME,
        "--template-file",
        "../infra/main.bicep",
        "--parameters",
        *parameters,
        "--no-wait",
    ]
    log("Bicep compile + preflight running (az deployment group create --no-wait)...")
    t0 = time.time()
    result = subprocess.run(cmd, capture_output=True, text=True)
    log(f"az create returned in {time.time() - t0:.1f}s (exit code {result.returncode})")
    return result


def get_deployment_state() -> str:
    result = subprocess.run(
        [
            az_cmd, "deployment", "group", "show",
            "-g", RESOURCE_GROUP, "-n", DEPLOYMENT_NAME,
            "--query", "properties.provisioningState", "-o", "tsv",
        ],
        capture_output=True,
        text=True,
    )
    return result.stdout.strip()


def list_operations() -> list:
    result = subprocess.run(
        [
            az_cmd, "deployment", "operation", "group", "list",
            "-g", RESOURCE_GROUP, "-n", DEPLOYMENT_NAME, "-o", "json",
        ],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0 or not result.stdout.strip():
        return []
    try:
        return json.loads(result.stdout)
    except json.JSONDecodeError:
        return []


def get_deployment_error() -> str:
    result = subprocess.run(
        [
            az_cmd, "deployment", "group", "show",
            "-g", RESOURCE_GROUP, "-n", DEPLOYMENT_NAME,
            "--query", "properties.error", "-o", "json",
        ],
        capture_output=True,
        text=True,
    )
    return result.stdout.strip()


def poll_until_done() -> str:
    seen: dict[str, str] = {}
    tick = 0
    while True:
        ops = list_operations()
        for op in ops:
            props = op.get("properties", {}) or {}
            target = props.get("targetResource") or {}
            resource_type = target.get("resourceType", "<unknown-type>")
            resource_name = target.get("resourceName", "<unknown-name>")
            op_state = props.get("provisioningState", "?")
            key = op.get("id") or f"{resource_type}/{resource_name}"
            if seen.get(key) != op_state:
                log(f"  [{op_state:<9}] {resource_type} / {resource_name}")
                seen[key] = op_state
        state = get_deployment_state()
        if state in ("Succeeded", "Failed", "Canceled"):
            return state
        tick += 1
        if tick % 6 == 0:  # heartbeat every ~30s even when nothing changed
            log(f"...still polling (deployment state: {state or 'not-yet-visible'}, ops seen: {len(seen)})")
        time.sleep(POLL_INTERVAL_SECONDS)


include_deployer_principal = bool(deployer_object_id)
retried_role_assignment = False
retried_after_purge = False

while True:
    log(f"Submitting deployment '{DEPLOYMENT_NAME}' (async)...")
    submit_result = submit_deployment(include_deployer_principal)
    submit_stderr = submit_result.stderr.strip()

    if submit_result.returncode != 0:
        # Preflight/validation errors surface here (e.g., DeploymentModelNotSupported).
        if (
            "RoleAssignmentExists" in submit_stderr
            and include_deployer_principal
            and not retried_role_assignment
        ):
            log("Detected existing AcrPush assignment. Retrying without deployerPrincipalId.")
            include_deployer_principal = False
            retried_role_assignment = True
            continue

        if "FlagMustBeSetForRestore" in submit_stderr and not retried_after_purge:
            soft_deleted_name = extract_soft_deleted_account_name(submit_stderr)
            if not soft_deleted_name:
                raise RuntimeError(
                    "Hit a soft-deleted AI Services account but could not parse its name.\n"
                    f"stderr: {submit_stderr}"
                )
            log(f"Purging soft-deleted account '{soft_deleted_name}' and retrying...")
            purge_result = subprocess.run(
                [
                    az_cmd, "cognitiveservices", "account", "purge",
                    "--name", soft_deleted_name,
                    "--location", LOCATION,
                    "--resource-group", RESOURCE_GROUP,
                ],
                capture_output=True,
                text=True,
            )
            if purge_result.returncode != 0:
                raise RuntimeError(
                    f"Failed to purge '{soft_deleted_name}'.\n"
                    f"stderr: {purge_result.stderr.strip()}"
                )
            retried_after_purge = True
            continue

        raise RuntimeError(
            f"Bicep deployment submission failed.\n"
            f"stderr: {submit_stderr}\n"
            f"stdout: {submit_result.stdout.strip()}"
        )

    log(f"ARM accepted the deployment. Polling per-resource progress every {POLL_INTERVAL_SECONDS}s...")
    final_state = poll_until_done()
    log(f"Deployment finished with state: {final_state}")

    if final_state == "Succeeded":
        break

    # Deployment failed post-submission — inspect the error and reapply retry heuristics.
    error_payload = get_deployment_error()

    if (
        "RoleAssignmentExists" in error_payload
        and include_deployer_principal
        and not retried_role_assignment
    ):
        log("Detected existing AcrPush assignment. Retrying without deployerPrincipalId.")
        include_deployer_principal = False
        retried_role_assignment = True
        continue

    if "FlagMustBeSetForRestore" in error_payload and not retried_after_purge:
        soft_deleted_name = extract_soft_deleted_account_name(error_payload)
        if not soft_deleted_name:
            raise RuntimeError(
                "Hit a soft-deleted AI Services account but could not parse its name.\n"
                f"error: {error_payload}"
            )
        log(f"Purging soft-deleted account '{soft_deleted_name}' and retrying...")
        purge_result = subprocess.run(
            [
                az_cmd, "cognitiveservices", "account", "purge",
                "--name", soft_deleted_name,
                "--location", LOCATION,
                "--resource-group", RESOURCE_GROUP,
            ],
            capture_output=True,
            text=True,
        )
        if purge_result.returncode != 0:
            raise RuntimeError(
                f"Failed to purge '{soft_deleted_name}'.\n"
                f"stderr: {purge_result.stderr.strip()}"
            )
        retried_after_purge = True
        continue

    raise RuntimeError(
        f"Bicep deployment failed with state '{final_state}'.\n"
        f"error: {error_payload}"
    )


## Step 4 - Retrieve deployment outputs

In [ ]:
import json
import subprocess

outputs_result = subprocess.run(
    [
        az_cmd,
        "deployment",
        "group",
        "list",
        "--resource-group",
        RESOURCE_GROUP,
        "--query",
        "[0].properties.outputs",
        "--output",
        "json",
    ],
    capture_output=True,
    text=True,
)
if outputs_result.returncode != 0:
    raise RuntimeError(
        f"az deployment group list failed (exit {outputs_result.returncode})\n"
        f"stderr: {outputs_result.stderr.strip()}\n"
        f"stdout: {outputs_result.stdout.strip()}"
    )
outputs = json.loads(outputs_result.stdout or "null")
if not outputs:
    raise RuntimeError("No deployment outputs found. Rerun Step 3 and check for errors.")

ai_account_name = outputs["aiServicesAccountName"]["value"]
ai_endpoint = outputs["aiServicesEndpoint"]["value"]
ai_deployment_name = outputs["aiDeploymentName"]["value"]
acr_name = outputs["acrName"]["value"]
acr_login_server = outputs["acrLoginServer"]["value"]
container_image_name = outputs["containerImageName"]["value"]
container_image_reference = outputs["containerImageReference"]["value"]
function_app_name = outputs["functionAppName"]["value"]
function_url = outputs["functionUrl"]["value"]
app_insights_name = outputs["appInsightsName"]["value"]
sample_repo_url = outputs["sampleRepoUrl"]["value"]

print(f"AI account         : {ai_account_name}")
print(f"AI endpoint        : {ai_endpoint}")
print(f"Model deployment   : {ai_deployment_name}")
print(f"ACR                : {acr_login_server}")
print(f"Container image    : {container_image_reference}")
print(f"Function app       : {function_app_name}")
print(f"Function URL       : {function_url}")
print(f"Sample repo        : {sample_repo_url}")

## Step 5 - Write outputs to `env/.env`

This persists values for the configure and test notebooks.

> **Security note:** `env/.env` is git-ignored. Never commit it.

In [ ]:
import os
from pathlib import Path

subscription_id = os.getenv("AZURE_SUBSCRIPTION_ID", "").strip()
if not subscription_id:
    sub_result = subprocess.run(
        [az_cmd, "account", "show", "--query", "id", "-o", "tsv"],
        capture_output=True,
        text=True,
    )
    if sub_result.returncode != 0:
        raise RuntimeError(
            f"az account show failed (exit {sub_result.returncode})\n"
            f"stderr: {sub_result.stderr.strip()}"
        )
    subscription_id = sub_result.stdout.strip()

env_lines = [
    "# Auto-generated by 01_deploy_infra.ipynb - do not commit",
    f"AZURE_SUBSCRIPTION_ID={subscription_id}",
    f"AZURE_RESOURCE_GROUP={RESOURCE_GROUP}",
    f"AZURE_AI_ENDPOINT={ai_endpoint}",
    f"AZURE_AI_ACCOUNT_NAME={ai_account_name}",
    f"AZURE_AI_DEPLOYMENT_NAME={ai_deployment_name}",
    f"AZURE_ACR_NAME={acr_name}",
    f"AZURE_ACR_LOGIN_SERVER={acr_login_server}",
    f"AZURE_CONTAINER_IMAGE_NAME={container_image_name}",
    f"AZURE_CONTAINER_IMAGE_REFERENCE={container_image_reference}",
    f"AZURE_FUNCTION_APP_NAME={function_app_name}",
    f"AZURE_FUNCTION_URL={function_url}",
    f"AZURE_APP_INSIGHTS_NAME={app_insights_name}",
    f"SAMPLE_REPO_URL={sample_repo_url}",
    "AZURE_FUNCTION_KEY=",
]

env_path = Path("../env/.env")
env_path.parent.mkdir(parents=True, exist_ok=True)
env_path.write_text("\n".join(env_lines) + "\n", encoding="utf-8")
print(f"Wrote {env_path.resolve()}")
print("AZURE_FUNCTION_KEY will be populated by notebook 02 after the container image is built.")

---

## Infrastructure deployed

Continue with **`02_configure.ipynb`** to build the container image in ACR and swap the Function App to it.